In [1]:
import numpy as np
import pandas as pd
from glob import glob
from pathlib import Path
from copy import deepcopy
from sklearn.decomposition import PCA
from cca_zoo.linear import CCA, TCCA
from scipy.sparse.linalg import eigsh
from joblib import Parallel, delayed
from collections import defaultdict
import re
import matplotlib.pyplot as plt

In [4]:

# -------------------------------
# 1. Data loading (full dataset)
# -------------------------------
root1 = '../../ESdB-simclr/log/full/'
root2 = '../../ESdB-2/log/full/'
d = 'age'
m1 = 'coles'
s1 = 'regression'
m2 = 'NTP_GRU'
s2 = 'forecast'

In [ ]:


def safe_stack(series):
    vals = []
    for x in series.values:
        if isinstance(x, tuple):
            x = x[0]
        arr = np.asarray(x.tolist() if hasattr(x, 'tolist') else x, dtype=np.float64)
        vals.append(arr)
    shapes = {a.shape for a in vals}
    if len(shapes) > 1:
        raise ValueError(f"Inconsistent shapes in column '{series.name}': {shapes}")
    return np.stack(vals)

X1_train = pd.read_parquet(Path(glob(f'{root1}/{d}/{m1}/tests/{s1}/seed_0/embeddings/train_postproc/')[0]))
X1_test  = pd.read_parquet(Path(glob(f'{root1}/{d}/{m1}/tests/{s1}/seed_0/embeddings/test_postproc/')[0]))
X2_train = pd.read_parquet(Path(glob(f'{root2}/{d}/{m2}/tests/{s2}/seed_0/embeddings/train_postproc/')[0]))
X2_test  = pd.read_parquet(Path(glob(f'{root2}/{d}/{m2}/tests/{s2}/seed_0/embeddings/test_postproc/')[0]))

global_emb1_train = safe_stack(X1_train['global_emb'])
global_emb2_train = safe_stack(X2_train['global_emb'])
#embeddings1_train = safe_stack(X1_train['embeddings'])
#embeddings2_train = safe_stack(X2_train['embeddings'])
shift_emb1_train  = safe_stack(X1_train['shift_emb'])
shift_emb2_train  = safe_stack(X2_train['shift_emb'])

global_emb1_test = safe_stack(X1_test['global_emb'])
global_emb2_test = safe_stack(X2_test['global_emb'])
#embeddings1_test = safe_stack(X1_test['embeddings'])
#embeddings2_test = safe_stack(X2_test['embeddings'])
shift_emb1_test  = safe_stack(X1_test['shift_emb'])
shift_emb2_test  = safe_stack(X2_test['shift_emb'])

print("Train shapes:")
for name, arr in [('global_emb1', global_emb1_train),
                  ('global_emb2', global_emb2_train),
                  ('shift_emb1', shift_emb1_train),
                  ('shift_emb2', shift_emb2_train)]:
    print(f"  {name}: {arr.shape}")

# Unified ranks: based on min feature dim (same for global and shift because both have 920 and 1024)
min_dim = min(global_emb1_train.shape[1], global_emb2_train.shape[1])
ranks = list(range(128, min_dim + 1, 128))
if min_dim % 128 != 0 and min_dim not in ranks:
    ranks.append(min_dim)
ranks = sorted(ranks)
print(f"Ranks: {ranks}")

# -------------------------------
# 2. Fusion functions (all 2D in, 2D out)
# -------------------------------
def _get_latent_dim(d1, d2, n_samples, user_dim):
    max_possible = min(d1, d2, n_samples)
    if user_dim is not None:
        return min(user_dim, max_possible)
    return max_possible

def concat_fusion(v1_train, v2_train, v1_test, v2_test):
    return np.concatenate([v1_train, v2_train], axis=1), np.concatenate([v1_test, v2_test], axis=1)

def pca_fusion(v1_train, v2_train, v1_test, v2_test, n_components=None):
    concat_train = np.concatenate([v1_train, v2_train], axis=1)
    concat_test  = np.concatenate([v1_test, v2_test], axis=1)
    d = _get_latent_dim(v1_train.shape[1], v2_train.shape[1], v1_train.shape[0], n_components)
    pca = PCA(n_components=d)
    return pca.fit_transform(concat_train), pca.transform(concat_test)

def cca_fusion(v1_train, v2_train, v1_test, v2_test, latent_dims=None):
    ld = _get_latent_dim(v1_train.shape[1], v2_train.shape[1], v1_train.shape[0], latent_dims)
    cca = CCA(latent_dimensions=ld).fit([v1_train, v2_train])
    tv = cca.transform([v1_train, v2_train])
    tev = cca.transform([v1_test, v2_test])
    return (tv[0] + tv[1]) / 2, (tev[0] + tev[1]) / 2

def tcca_fusion(v1_train, v2_train, v1_test, v2_test, latent_dims=None):
    raise RuntimeError("TCCA skipped due to memory constraints")

def dcca_fusion(v1_train, v2_train, v1_test, v2_test, latent_dims=None):
    raise RuntimeError("DCCA not yet implemented")

def dtcca_fusion(v1_train, v2_train, v1_test, v2_test, latent_dims=None):
    raise RuntimeError("DTCCA not yet implemented")

# -------------------------------
# Efficient Tucker (fit / transform separated, parallelised)
# -------------------------------
class EfficientTucker:
    def __init__(self, rank=None, batch_size=512, n_jobs=-1):
        self.rank = rank
        self.batch_size = batch_size
        self.n_jobs = n_jobs
        self.B_ = None
        self.C_ = None

    def fit(self, v1_train, v2_train):
        F1, F2 = v1_train.shape[1], v2_train.shape[1]
        max_rank = min(F1, F2)
        if self.rank is None:
            r = max_rank
        else:
            r = min(self.rank[0] if isinstance(self.rank, tuple) else self.rank, max_rank)

        N = len(v1_train)
        batches = [(v1_train[i:i+self.batch_size], v2_train[i:i+self.batch_size])
                   for i in range(0, N, self.batch_size)]

        def process_batch(bv1, bv2):
            G1 = np.zeros((F1, F1))
            G2 = np.zeros((F2, F2))
            for x, y in zip(bv1, bv2):
                G1 += np.outer(x, x) * (y @ y)
                G2 += np.outer(y, y) * (x @ x)
            return G1, G2

        gram_parts = Parallel(n_jobs=self.n_jobs, backend='threading')(
            delayed(process_batch)(bv1, bv2) for bv1, bv2 in batches
        )
        G1 = np.sum(np.array([g[0] for g in gram_parts]), axis=0)
        G2 = np.sum(np.array([g[1] for g in gram_parts]), axis=0)

        _, B = eigsh(G1, k=r, which='LM')
        _, C = eigsh(G2, k=r, which='LM')
        self.B_ = B[:, ::-1]
        self.C_ = C[:, ::-1]
        self.r_ = r
        return self

    def transform(self, v1, v2):
        if self.B_ is None or self.C_ is None:
            raise RuntimeError("Fit the model first.")
        proj1 = v1 @ self.B_
        proj2 = v2 @ self.C_
        return np.concatenate([proj1, proj2], axis=1)

    def fit_transform(self, v1_train, v2_train, v1_test, v2_test):
        self.fit(v1_train, v2_train)
        return self.transform(v1_train, v2_train), self.transform(v1_test, v2_test)

def tucker_fusion_efficient(v1_train, v2_train, v1_test, v2_test, rank=None):
    return EfficientTucker(rank=rank).fit_transform(v1_train, v2_train, v1_test, v2_test)

# -------------------------------
# 3. 3D fusion (flatten time, fit, reshape)
# -------------------------------
def fuse_3d(fusion_func, v1_train, v2_train, v1_test, v2_test, name="3d"):
    B_tr, T_tr, F1 = v1_train.shape
    B_te, T_te, _ = v1_test.shape

    flat1_tr = v1_train.reshape(-1, F1)
    flat2_tr = v2_train.reshape(-1, v2_train.shape[2])
    flat1_te = v1_test.reshape(-1, F1)
    flat2_te = v2_test.reshape(-1, v2_test.shape[2])

    print(f"    [{name}] Flattened train: {flat1_tr.shape[0]} samples")
    fused_flat_tr, fused_flat_te = fusion_func(flat1_tr, flat2_tr, flat1_te, flat2_te)

    fused_dim = fused_flat_tr.shape[1]
    print(f"    [{name}] Fused dimension: {fused_dim}")

    fused_tr = fused_flat_tr.reshape(B_tr, T_tr, fused_dim)
    fused_te = fused_flat_te.reshape(B_te, T_te, fused_dim)
    return fused_tr, fused_te

# -------------------------------
# 4. Process one method (with rank-aware callables)
# -------------------------------
def process_method(method_name, fusion_func_global, fusion_func_shift, rank=None):
    rank_str = f"_rank{rank}" if rank is not None else ""
    print(f"\n=== {method_name}{rank_str} ===")
    try:
        # 2D global embeddings
        g_tr, g_te = fusion_func_global(global_emb1_train, global_emb2_train,
                                        global_emb1_test, global_emb2_test)
        print(f"  global_emb fused dim: {g_tr.shape[1]}")

        # 3D shift embeddings
        s_tr, s_te = fuse_3d(fusion_func_shift,
                             shift_emb1_train, shift_emb2_train,
                             shift_emb1_test, shift_emb2_test,
                             name="shift_emb")
        print(f"  shift_emb fused dim: {s_tr.shape[2]}")

        # Save
        out_train = deepcopy(X1_train)
        out_test  = deepcopy(X1_test)
        out_train['global_emb'] = [x.tolist() for x in g_tr]
        out_test['global_emb']  = [x.tolist() for x in g_te]
        out_train['shift_emb']  = [x.tolist() for x in s_tr]
        out_test['shift_emb']   = [x.tolist() for x in s_te]

        out_dir = Path(root2) / d / m2 / 'tests' / s2 / 'seed_0' / f'embeddings_{method_name}{rank_str}'
        (out_dir / 'train_postproc').mkdir(parents=True, exist_ok=True)
        (out_dir / 'test_postproc').mkdir(parents=True, exist_ok=True)
        out_train.to_parquet(out_dir / 'train_postproc' / 'data.parquet')
        out_test.to_parquet(out_dir / 'test_postproc' / 'data.parquet')
        print(f"  Saved to {out_dir}")

    except Exception as e:
        print(f"  SKIPPED: {e}")

# -------------------------------
# 5. Run methods with rank sweeps
# -------------------------------
# Concatenation (once, no rank)
process_method('concatenation', fusion_func_global=concat_fusion, fusion_func_shift=concat_fusion)

# PCA, CCA, Tucker for each rank
for rank in ranks:
    # PCA
    pca_global = lambda v1, v2, t1, t2: pca_fusion(v1, v2, t1, t2, n_components=rank)
    pca_shift  = lambda v1, v2, t1, t2: pca_fusion(v1, v2, t1, t2, n_components=rank)
    process_method('PCA', fusion_func_global=pca_global, fusion_func_shift=pca_shift, rank=rank)

    # CCA
    cca_global = lambda v1, v2, t1, t2: cca_fusion(v1, v2, t1, t2, latent_dims=rank)
    cca_shift  = lambda v1, v2, t1, t2: cca_fusion(v1, v2, t1, t2, latent_dims=rank)
    process_method('CCA', fusion_func_global=cca_global, fusion_func_shift=cca_shift, rank=rank)

    # Tucker
    tucker_global = lambda v1, v2, t1, t2: tucker_fusion_efficient(v1, v2, t1, t2, rank=rank)
    tucker_shift  = lambda v1, v2, t1, t2: tucker_fusion_efficient(v1, v2, t1, t2, rank=rank)
    process_method('Tucker', fusion_func_global=tucker_global, fusion_func_shift=tucker_shift, rank=rank)

print("\nAll available methods completed.")

Train shapes:
  global_emb1: (30000, 920)
  global_emb2: (30000, 1024)
  shift_emb1: (30000, 9, 920)
  shift_emb2: (30000, 9, 1024)
Ranks: [128, 256, 384, 512, 640, 768, 896, 920]

=== concatenation ===
  global_emb fused dim: 1944
    [shift_emb] Flattened train: 270000 samples
    [shift_emb] Fused dimension: 1944
  shift_emb fused dim: 1944
  Saved to ../../ESdB-2/log/full/age/NTP_GRU/tests/forecast/seed_0/embeddings_concatenation

=== PCA_rank128 ===
  global_emb fused dim: 128
    [shift_emb] Flattened train: 270000 samples
    [shift_emb] Fused dimension: 128
  shift_emb fused dim: 128
  Saved to ../../ESdB-2/log/full/age/NTP_GRU/tests/forecast/seed_0/embeddings_PCA_rank128

=== CCA_rank128 ===
  global_emb fused dim: 128
    [shift_emb] Flattened train: 270000 samples
    [shift_emb] Fused dimension: 128
  shift_emb fused dim: 128
  Saved to ../../ESdB-2/log/full/age/NTP_GRU/tests/forecast/seed_0/embeddings_CCA_rank128

=== Tucker_rank128 ===
  global_emb fused dim: 256
    [shi

In [ ]:
assert

In [ ]:
# res

In [ ]:
#W = glob('../validator_output_2026-07-03_17-37-32_embeddings_concatenation/*/')
W = glob('../validator_output_*/')
t = lambda: defaultdict(t)
for w in sorted(W):
    csvs = glob(f"{w}/*.csv")
    if csvs:
        all_metrics = pd.concat([pd.read_csv(csv) for csv in csvs])
        avg_per_metric = all_metrics.groupby('metric')['value'].mean()
        print(w)
        print(avg_per_metric)
        

In [ ]:
W = glob('../result_validator_output/validator_output_age_ntp_gru_S*/*')
data = []
df_result_emb1 = pd.read_csv(f'../log_age_ntp_gru_SimCLR/full/{d}/{m2}/tests/{s2}/results.csv')

for w in sorted(W):
    m = re.search(r'embeddings_(\w+)', w)
    if not m: continue
    postfix = m.group(1)
    if '_rank' in postfix:
        method, rank_str = postfix.rsplit('_rank', 1)
        
        rank = int("".join([str(e) for e in rank_str if e.isnumeric()]))
    else:
        method, rank = postfix, None
    csvs = glob(f"{w}/*.csv")
    if not csvs: continue
    all_metrics = pd.concat([pd.read_csv(c) for c in csvs])
    avg = all_metrics.groupby('metric')['value'].mean()
    for metric_name, val in avg.items():
        task = metric_name.split('__')[1]
        data.append((method, rank, task, val))

df = pd.DataFrame(data, columns=['method','rank','task','value'])
max_rank = df['rank'].max()
concat_x = max_rank + 128
df.loc[df['method'] == 'concatenation', 'rank'] = concat_x

tasks = df['task'].unique()
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

ref = {
    'age': {'SimCLR': 0.690, 'NTP_GRU': 0.683},
    'reg_amount': {'SimCLR': 0.506, 'NTP_GRU': 0.463},
    'forecast': {'SimCLR': 0.344, 'NTP_GRU': 0.411},
    'anomaly': {'SimCLR': 0.800, 'NTP_GRU': 0.745}
}

for i, task in enumerate(tasks):
    ax = axes[i]
    task_df = df[df['task'] == task]
    for method, grp in task_df[task_df['method'] != 'concatenation'].groupby('method'):
        grp = grp.sort_values('rank')
        ax.plot(grp['rank'], grp['value'], marker='o', label=method)
    concat_grp = task_df[task_df['method'] == 'concatenation']
    if not concat_grp.empty:
        ax.scatter(concat_grp['rank'], concat_grp['value'], 
                   color='black', s=80, marker='D', label='concatenation', zorder=5)
    if task in ref:
        ax.axhline(ref[task]['SimCLR'], color='C0', linestyle='--', alpha=0.7, label='SimCLR (single)')
        ax.axhline(ref[task]['NTP_GRU'], color='C1', linestyle='--', alpha=0.7, label='NTP_GRU (single)')
    ax.set_title(task)
    ax.set_xlabel('Rank')
    ax.set_ylabel('Metric')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [7]:
df

NameError: name 'df' is not defined